In [1]:
with open("smartprix.html" ,mode='r',encoding='utf-8') as f:
  html=f.read()
from bs4 import BeautifulSoup
soup=BeautifulSoup(html,'lxml')
len=len(soup.find_all('div',{'class':'sm-product has-tag has-features has-actions'}))

In [18]:
import re
import pandas as pd
import numpy as np

container = soup.find_all('div', {'class': 'sm-product has-tag has-features has-actions'})
rows = []

# Improved and more robust patterns
patterns = {
    'sim': r'\bsim\b|dual|single',
    'processor': r'\bprocessor\b|\bcore\b|\bsnapdragon\b|\bdimensity\b|\bhelio\b|\bexynos\b|\btensor\b|\bbionic\b|\bunisoc\b|\bkirin\b|\bchip\b',
    'ram': r'\bram\b|\binbuilt\b|\bgibibyte\b|gb\b',
    'battery': r'\bbattery\b|\bmah\b|\thelium\b',
    'display': r'\bdisplay\b|\binches\b|\bpx\b|["“”]|\bhz\b|\boled\b|\blcd\b|\bamoled\b',
    'camera': r'\bcamera\b|\bfront\b|\brear\b|\bmp\b',
    'card': r'\bmemory card\b|\bcard\b|\bexpandable\b',
    'os': r'\bandroid\b|\bios\b|\bs30\+|\bkaios\b|\bharmonyos\b|\bhyperos\b|\bcoloros\b|\boxygenos\b|\bfuntouch\b|\bone ui\b|\brealme ui\b|\bmagicos\b|\boriginos\b|\bnothing os\b|\bmyos\b'}

for i in container:
    name = i.find('h2').text.strip()
    price = i.find('span', {'class': 'price'}).text.strip()

    try:
        score_element = i.select_one('div.score b')
        score = score_element.get_text(strip=True) if score_element else np.nan
    except:
        score = np.nan

    row = {
        'name': name,
        'price': price,
        'rating': score
    }

    # Get all specifications as lowercase text to perform accurate regex matching
    spec_elements = i.find('ul', {'class': 'sm-feat specs'}).find_all('li')
    spec_list = [s.get_text(strip=True) for s in spec_elements]

    for field, pattern in patterns.items():
        matched_value = np.nan
        for spec in spec_list:
            if re.search(pattern, spec, re.IGNORECASE):
                matched_value = spec
                break
        row[field] = matched_value

    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv('smartprix.csv', index=False)

In [19]:
df.isnull().sum()

,0
name,0
price,0
rating,0
sim,10
processor,43
ram,1
battery,0
display,0
camera,3
card,261


In [20]:
%pip install pandas sqlalchemy "psycopg[binary]"
from sqlalchemy import create_engine
from google.colab import userdata

database_url=userdata.get('database_url')
engine=create_engine(database_url)
df.to_sql(
    name='mobile_scraped'
    ,con=engine
    ,index=False
    ,if_exists='replace'
)
print("data uploaded successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 24.0 MB/s eta 0:00:00
data uploaded successfully


In [23]:
read=pd.read_sql('select * from mobile_scraped',engine)
read


,name,price,rating,sim,processor,ram,battery,display,camera,card,os
0,OPPO K14 Plus,"₹25,999",75,"Dual Sim, 5G, VoLTE, Wi-Fi, IR Blaster","Dimensity 7360 Max, Octa Core, 2.5 GHz Processor","6 GB RAM, 128 GB inbuilt",8000 mAh Battery with 45W Fast Charging,"6.78 inches, 1272 x 2772 px, 144 Hz Display wi...",50 MP + 2 MP Dual Rear & 16 MP Front Camera,Memory Card Not Supported,Android v16
1,Samsung Galaxy S25 Ultra,"₹94,999",92,"Dual Sim, 5G, VoLTE, Vo5G, NFC, eSIM, Wi-Fi","Snapdragon 8 Elite for Galaxy, Octa Core, 4.47...","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 45W Fast Charging,"6.9 inches, 1440 x 3120 px, 120 Hz Display wit...",200 MP Quad Rear & 12 MP Front Camera,Memory Card Not Supported,Android v15
2,OPPO K14 Plus,"₹25,999",75,"Dual Sim, 5G, VoLTE, Wi-Fi, IR Blaster","Dimensity 7360 Max, Octa Core, 2.5 GHz Processor","6 GB RAM, 128 GB inbuilt",8000 mAh Battery with 45W Fast Charging,"6.78 inches, 1272 x 2772 px, 144 Hz Display wi...",50 MP + 2 MP Dual Rear & 16 MP Front Camera,Memory Card Not Supported,Android v16
3,Vivo V80 5G,"₹62,999",84,"Dual Sim, 5G, VoLTE, NFC, Wi-Fi, IR Blaster","Snapdragon 7 Gen4, Octa Core, 2.8 GHz Processor","8 GB RAM, 128 GB inbuilt",7200 mAh Battery with 90W Fast Charging,"6.59 inches, 1260 x 2750 px, 144 Hz Display wi...",50 MP + 50 MP + 8 MP Triple Rear & 50 MP Front...,Memory Card Not Supported,Android v17
4,Motorola Edge 70 Fusion,"₹29,390",70,"Dual Sim, 5G, VoLTE, Vo5G, NFC, Wi-Fi","Snapdragon 7s Gen4, Octa Core, 2.7 GHz Processor","8 GB RAM, 128 GB inbuilt",7000 mAh Battery with 68W Fast Charging,"6.8 inches, 1272 x 2772 px, 144 Hz Display wit...",50 MP + 13 MP + Dedicated 3-in-1 light sensor ...,None,Android v16
...,...,...,...,...,...,...,...,...,...,...,...
981,Snexian Bold V1,"₹2,299",9,Dual Sim,1.77 MHz Processor,"32 MB RAM, 32 MB inbuilt",1000 mAh Battery,"1.8 inches, 320 x 420 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 16 GB",None
982,Micromax X1i Smart,"₹1,499",8,Dual Sim,"Single Core, 1.2 MHz Processor","32 MB RAM, 32 MB inbuilt",1200 mAh Battery,"2.4 inches, 320 x 240 px Display",0.08 MP Rear Camera,Memory Card Supported,None
983,Itel it5262,"₹1,579",10,Dual Sim,None,"4 MB RAM, 4 MB inbuilt",2000 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB",None
984,Poco C65 (6GB RAM + 128GB),"₹19,999",48,"Dual Sim, 4G, VoLTE, Wi-Fi","Helio G85, Octa Core, 2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 18W Fast Charging,"6.74 inches, 720 x 1600 px, 90 Hz Display with...",50 MP + 2 MP Triple Rear & 8 MP Front Camera,"Memory Card Supported, upto 1 TB",Android v13
